## Item 5 — "Destination": on a retry round, run retrieval only for the books `reflect` labeled `miss`; leave every other book's slot untouched.
*Budget: ~5 landed units expected.*

```mermaid
flowchart LR
    subgraph NOW["today"]
        R1["reflect: A ok, B miss"] --> RT1["retriever runs for A and B"]
    end
    subgraph DEST["destination"]
        R2["reflect: A ok, B miss"] --> RT2["retriever runs for B only"]
        R2 -.-> KEEP["A's slot stays as it was"]
    end
    NOW ==> DEST
```

In [ ]:
# 1 - "whether to retry" (the gate) and "which books to retry" (the retriever's loop) are two separate decisions, and that only the first has any per-book logic today?

# 2 - the verdict the retriever needs (which books to redo) is already in each book's slot when it starts, so the gate doesn't have to pass anything to it?

# 3 - a blank round-1 slot and a settled ok/silent slot both lack a narrowed_query, so the skip rule has to read label, not narrowed_query?

# 4 - A book omitted from the return is deleted from state, and that reflect would crash on it rather than send it back for retrieval?

# 5 - the composer loop overwrites one prompt variable on every pass, so the book visited last decides the prompt?

## Item 6.1 — "Destination": `composer` returns structured slots — agreement, disagreement, and each corpus's unique claims — instead of one blended paragraph; a corpus `reflect` has labeled "silent" gets an explicit absence statement in its own slot, never omitted or folded into hedged prose.
*Budget: ~4 landed units expected.*

```mermaid
flowchart LR
    subgraph NOW["today"]
        C1["composer: one blended paragraph (answer: str)"]
    end
    subgraph DEST["destination"]
        C2["composer: structured slots - agreement / disagreement / per-corpus unique claims"]
        C2 -.-> SIL["a corpus labeled silent gets an explicit absence statement, not omitted"]
    end
    NOW ==> DEST
```

In [ ]:
# Landed 1 — composer returns one blended paragraph from a single llm.invoke call (agent.py:178-189); reflect already critiques per corpus_id (agent.py:211-272) but has nothing per-book shaped in the answer to check against.
# Landed 2 — the new shape splits into one cross-book pair (agreement/disagreement) plus a per-corpus list (corpus_id, claim), mirroring the top+list nesting reflect's Critique/CorpusCritique already uses (agent.py:64-71).
# Landed 3 — composer's silent-override reuses reflect's alibi mechanic (label as the after-the-call trigger for a code overwrite, not trusting the model's prose): if label == "silent", overwrite that entry's claim in code, same shape as reflect's own narrowed_query guard (agent.py:237-238), not reflect's literal code.
# Landed 4 — the call keeps today's needs_revision branch (agent.py:183-186) and return shape; the deltas are an added book_map/label string in the prompt and answer becoming the structured object instead of a string. Parked: reflect's prompt still reads state['answer'] as a string (agent.py:225) — that's 6.2's own Destination.